# Classification boundaries
Synthetic two-moons data makes model geometry visible. The aim is to connect hyperparameters with the shape and complexity of a decision rule.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=600, noise=.25, random_state=42)
cv = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    'tree_depth_3': DecisionTreeClassifier(max_depth=3, random_state=42),
    'tree_unlimited': DecisionTreeClassifier(random_state=42),
    'knn_5': make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    'knn_25': make_pipeline(StandardScaler(), KNeighborsClassifier(25)),
}
scores = {name: cross_val_score(model, X, y, cv=cv).mean() for name, model in models.items()}
pd.Series(scores, name='cv_accuracy').sort_values(ascending=False).round(3)


In [ ]:
model = models[max(scores, key=scores.get)]
model.fit(X, y)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 220), np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 220))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.figure(figsize=(7,5))
plt.contourf(xx, yy, Z, alpha=.25)
plt.scatter(X[:,0], X[:,1], c=y, s=12)
plt.title(f'Best CV model: {max(scores, key=scores.get)}')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()


## Interpretation
Small neighborhoods and unlimited trees can create highly irregular boundaries. Cross-validation lets us prefer complexity only when it survives resampling. A visually intricate boundary is not evidence of a better model by itself.